# Notebook 19 — Geographic discovery and reviewed local media

**Fictional, offline demonstration.** This tutorial uses a labeled synthetic OSM fixture, a generated PNG/JPEG and made-up review statements. It demonstrates *system mechanics*, not geolocation accuracy, real licensing, menu freshness or retrieval effectiveness. No network access or third-party media download occurs.

In [ ]:
import hashlib
import json
import tempfile
from pathlib import Path

from PIL import Image
from fastapi.testclient import TestClient

from savoroute.api.app import create_app
from savoroute.data.catalog import catalog_to_dict, load_catalog
from savoroute.evidence.review import catalog_digest, prepare_review, publish_evidence
from savoroute.geography.nearby import NearbyIndex
from savoroute.ingestion.osm import import_osm_snapshot
from savoroute.media.reviewed import package_reviewed_media, load_reviewed_media

ROOT = Path.cwd()
assert (ROOT / "data/sample/osm_overpass_fixture_versioned.json").exists()


## 1. Bind geographic coordinates without changing the canonical catalog

The OSM fixture contains fictional locations. Only admitted identities with saved geometry become searchable. Ways/relations use the Overpass bounding-box center, not the door location.

In [ ]:
osm = json.loads((ROOT / "data/sample/osm_overpass_fixture_versioned.json").read_text())
result = import_osm_snapshot(osm, synthetic_fixture=True)
nearby = NearbyIndex.from_snapshot(result.catalog, osm)
search_results = nearby.search(42.35, -71.06, radius_m=2000)
assert search_results
[(hit["item"].title, hit["distance_m"], hit["geometry_basis"]) for hit in search_results]


## 2. Explicit, fictitious local image review and publication

A local file hash proves integrity against a declared local file. It cannot prove external provenance or permission. The synthetic `example.org` claim below is for software tests only; do not publish it as a real photograph.

In [ ]:
tmp = tempfile.TemporaryDirectory()
work = Path(tmp.name)
catalog = load_catalog()
image = work / "test-image.jpg"
Image.new("RGB", (160, 120), (184, 96, 61)).save(image, "JPEG")
proposals = work / "proposals.json"
review = work / "review.json"
approved = work / "approved.json"
manifest = work / "local-files.json"
proposals.write_text(json.dumps({
    "schema_version": 1,
    "catalog_digest": catalog_digest(catalog),
    "claims": [
        {"claim_id": "demo_image", "item_id": "r001", "kind": "image_url",
         "value": "https://example.org/fictional-photo", "source_url": "https://example.org/fictional-photo",
         "observed_on": "2026-01-01", "license": "CC-BY-4.0", "attribution": "Fictional example"},
        {"claim_id": "demo_menu", "item_id": "r001", "kind": "menu_url",
         "value": "https://example.org/fictional-menu", "source_url": "https://example.org/fictional-menu",
         "observed_on": "2026-01-01", "license": None, "attribution": None},
    ],
}, indent=2))
assert prepare_review(catalog, proposals, review) == 2
decisions = json.loads(review.read_text())
for entry in decisions["decisions"]:
    entry.update(decision="approve", reviewer="fictional-demo", reason="Fictional example only",
                 reviewed_on="2026-01-02")
review.write_text(json.dumps(decisions))
publication = publish_evidence(catalog, proposals, review, approved)
assert len(publication["claims"]) == 2


In [ ]:
manifest.write_text(json.dumps({
    "schema_version": 1, "catalog_digest": catalog_digest(catalog),
    "assets": [{"claim_id": "demo_image", "path": "test-image.jpg",
                "sha256": hashlib.sha256(image.read_bytes()).hexdigest()}],
}, indent=2))
asset_root = work / "packaged-media"
bundle = package_reviewed_media(catalog, approved, manifest, asset_root)
assert load_reviewed_media(asset_root, catalog, publication) == bundle
asset = bundle["assets"][0]
assert "metadata removed" in asset["modifications"]
(asset["asset_id"], asset["license"], asset["sha256"])


## 3. Reuse the HTTP API, with distinct menu status and media attribution

The local FastAPI client does not open a socket. The same server may additionally be configured with `--osm` for nearby results. The reviewer identifier/rationale do not appear in public media metadata.

In [ ]:
catalog_file = work / "catalog.json"
catalog_file.write_text(json.dumps(catalog_to_dict(catalog)))
with TestClient(create_app(catalog_path=catalog_file, evidence_path=approved,
                           reviewed_media_path=asset_root)) as client:
    meta = client.get("/v1/meta").json()
    menu = client.get("/v1/items/r001/evidence").json()
    unknown = client.get("/v1/items/r002/evidence").json()
    media = client.get("/v1/items/r001/media").json()
    png = client.get(f"/v1/media/{asset['asset_id']}")
    assert meta["reviewed_media_enabled"]
    assert menu["menu_status"] == "reviewed_reference"
    assert unknown["menu_status"] == "not_recorded"
    assert media["images"][0]["attribution"] == "Fictional example"
    assert png.status_code == 200 and png.content.startswith(b"\x89PNG")
    assert "review" not in media["images"][0]
(menu["menu_status"], unknown["menu_status"], png.status_code)


## 4. Boundaries

- Nearby ranks are straight-line distances to recorded geometry, not routes or visit probabilities.
- OpenStreetMap data reuse requires ODbL attribution. This does **not** cover rights in third-party photographs.
- Licensed/approved metadata does **not** independently prove the right to publish bytes. Confirm source content and license terms outside this example.
- `menu_status="not_recorded"` means missing reviewed reference, not absence of a real menu.
- The existing unannotated OSM relevance study provides no quality estimate for these features.